# JED Multi-Step Tool Attack — submission

Verify-and-keep red-team attack targeting the two predicates reachable under the public OptimalGuardrail: EXFILTRATION (http.post to a clean host, secret in the body) and CONFUSED_DEPUTY (email.send via the messaging tool, avoiding share words). Multi-endpoint messages fire up to 8 posts per turn; candidates are individually verified live so replay wastes no budget.

In [ ]:
import sys, os, glob
from pathlib import Path
sys.argv = [sys.argv[0]]
for candidate in glob.glob('/kaggle/input/**/kaggle_evaluation', recursive=True):
    root = str(Path(candidate).parent)
    if root not in sys.path:
        sys.path.insert(0, root)
    print('Dataset root:', root)
    break
else:
    raise RuntimeError('Attach the competition data source first.')
sys.path.insert(0, '/kaggle/working')
print('Setup complete')


In [ ]:
import base64
ATTACK_B64 = '...'
with open('/kaggle/working/attack.py', 'wb') as f:
    f.write(base64.b64decode(ATTACK_B64))
print('Wrote /kaggle/working/attack.py')


## Run the gateway
Starts the inference server. During the competition rerun the gateway drives it against gpt-oss and gemma and writes `submission.csv`.

In [ ]:
import kaggle_evaluation.jed_attack_134815.jed_attack_inference_server as srv
srv.JEDAttackInferenceServer().serve()
